# ElasticNet Regression on Walmart Weekly Sales - Technical Paper Version

This notebook implements a rigorous ElasticNet Regression analysis suitable for academic/technical papers:
1) Loading and preprocessing the Walmart sales dataset
2) Feature engineering with domain knowledge
3) 60% validation, 20% training, 20% test split
4) Systematic hyperparameter optimization using cross-validation
5) Comprehensive evaluation on train, validation, and test sets
6) Metrics reported in interpretable units (dollars)

ElasticNet Regression:
- Linear model with combined L1 (Lasso) and L2 (Ridge) penalties
- Balances sparsity (feature selection) and stability
- Key hyperparameters: alpha (regularization strength), l1_ratio (L1 vs L2 mix), max_iter
- Hyperparameters optimized via 5-fold cross-validation
- Scaling recommended for linear models

In [ ]:
# 1) Imports and reproducibility
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.linear_model import ElasticNet

np.random.seed(42)  # reproducibility

In [ ]:
# 2) Load the dataset and parse dates safely
# Try local relative path first; if it fails, try the Downloads path
csv_candidates = [
    'walmart-sales-dataset-of-45stores.csv',
    '/Users/jaacabrera/Downloads/walmart-sales-dataset-of-45stores.csv',
]

df = None
for p in csv_candidates:
    try:
        df = pd.read_csv(p, low_memory=False)
        print(f'Loaded: {p}')
        break
    except Exception:
        pass

if df is None:
    raise FileNotFoundError('Could not find the dataset. Update the path in csv_candidates.')

# Parse Date with day-first format (file has dd-mm-yyyy like "19-02-2010")
df['Date'] = pd.to_datetime(df['Date'].astype('string'), errors='coerce', dayfirst=True)
bad_dates = df['Date'].isna().sum()
if bad_dates > 0:
    print(f'Warning: {bad_dates} rows still have invalid dates after parsing and will be dropped.')
    df = df.dropna(subset=['Date']).reset_index(drop=True)

# Drop rows with missing target just in case
if 'Weekly_Sales' not in df.columns:
    raise KeyError('Weekly_Sales column not found. Check your CSV headers.')
df = df.dropna(subset=['Weekly_Sales']).reset_index(drop=True)

print(f'Data shape after date/target checks: {df.shape}')
df.head(3)

In [ ]:
# 3) Create simple, beginner-friendly features (no leakage from the target)
# Date-derived features
df['Month'] = df['Date'].dt.month
df['DayOfWeek'] = df['Date'].dt.dayofweek
df['Week'] = df['Date'].dt.isocalendar().week.astype(int)
df['Quarter'] = df['Date'].dt.quarter
df['IsWeekend'] = (df['DayOfWeek'] >= 5).astype(int)

# Store encoded as numeric (already numeric in this dataset)
if 'Store' in df.columns:
    df['Store_Encoded'] = df['Store']
else:
    df['Store_Encoded'] = 0  # fallback if missing

# Choose a small, readable feature set
feature_cols = [
    'Holiday_Flag', 'Temperature', 'Fuel_Price', 'CPI', 'Unemployment',
    'Store_Encoded', 'Month', 'DayOfWeek', 'Week', 'Quarter', 'IsWeekend'
]

# Build X (features) and y (target)
X = df[feature_cols].copy()
y = df['Weekly_Sales'].copy()

print('X shape:', X.shape, '| y shape:', y.shape)
print('Features:', feature_cols)

In [ ]:
# 4) Split into 60% validation, 20% training, 20% test (as requested)
# Stage 1: 20% test holdout
X_rem, X_test, y_rem, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

# Stage 2: Of the remaining 80%, use 25% for training (overall 20%) and 75% for validation (overall 60%)
X_train, X_val, y_train, y_val = train_test_split(X_rem, y_rem, train_size=0.25, random_state=42)

print(f"Split sizes -> train: {len(X_train)} ({len(y_train)/len(y):.0%}), val: {len(X_val)} ({len(y_val)/len(y):.0%}), test: {len(X_test)} ({len(y_test)/len(y):.0%})")

In [ ]:
# 5) Systematic hyperparameter optimization using cross-validation
# Linear models benefit from scaling

imputer = SimpleImputer(strategy='median')

# Fit imputer on TRAIN only, then apply to VAL/TEST (prevents data leakage)
X_train_imp = pd.DataFrame(imputer.fit_transform(X_train), columns=feature_cols)
X_val_imp   = pd.DataFrame(imputer.transform(X_val), columns=feature_cols)
X_test_imp  = pd.DataFrame(imputer.transform(X_test), columns=feature_cols)

# Scale
scaler = StandardScaler()
X_train_final = scaler.fit_transform(X_train_imp)
X_val_final   = scaler.transform(X_val_imp)
X_test_final  = scaler.transform(X_test_imp)

print('Performing 5-fold cross-validation to find optimal hyperparameters...')
print('Tuning: alpha, l1_ratio, max_iter')
print('This may take a few minutes (reduced grid for faster tuning)...\n')

# Reduced parameter grid for faster tuning
param_grid = {
    'alpha': [0.1, 1.0],
    'l1_ratio': [0.2, 0.5, 0.8],
    'max_iter': [2000],
    'random_state': [42]
}

elastic_cv = GridSearchCV(
    estimator=ElasticNet(),
    param_grid=param_grid,
    cv=5,
    scoring='r2',
    n_jobs=-1,
    verbose=1
)
elastic_cv.fit(X_train_final, y_train)

# Extract best model and parameters
elastic = elastic_cv.best_estimator_
best_params = elastic_cv.best_params_
cv_score = elastic_cv.best_score_

print(f"\n{'='*70}")
print('CROSS-VALIDATION RESULTS')
print(f"{'='*70}")
print(f"Best alpha: {best_params['alpha']}")
print(f"Best l1_ratio: {best_params['l1_ratio']}")
print(f"Best max_iter: {best_params['max_iter']}")
print(f"CV R² score (5-fold): {cv_score:.4f}")
print(f"{'='*70}\n")

# Predict on all sets
preds_train = elastic.predict(X_train_final)
preds_val = elastic.predict(X_val_final)
preds_test = elastic.predict(X_test_final)

# Calculate metrics
def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

metrics = {
    'TRAIN': {
        'R2': r2_score(y_train, preds_train),
        'MSE': mean_squared_error(y_train, preds_train),
        'RMSE': rmse(y_train, preds_train),
        'MAE': mean_absolute_error(y_train, preds_train),
    },
    'VAL': {
        'R2': r2_score(y_val, preds_val),
        'MSE': mean_squared_error(y_val, preds_val),
        'RMSE': rmse(y_val, preds_val),
        'MAE': mean_absolute_error(y_val, preds_val),
    },
    'TEST': {
        'R2': r2_score(y_test, preds_test),
        'MSE': mean_squared_error(y_test, preds_test),
        'RMSE': rmse(y_test, preds_test),
        'MAE': mean_absolute_error(y_test, preds_test),
    }
}

print('ElasticNet Regression Performance (hyperparameters optimized via CV)')
print(f"  Train Metrics       -> R²: {metrics['TRAIN']['R2']:.4f} | RMSE: ${metrics['TRAIN']['RMSE']:,.2f} | MAE: ${metrics['TRAIN']['MAE']:,.2f}")
print(f"  Validation Metrics  -> R²: {metrics['VAL']['R2']:.4f} | RMSE: ${metrics['VAL']['RMSE']:,.2f} | MAE: ${metrics['VAL']['MAE']:,.2f}")
print(f"  Test Metrics        -> R²: {metrics['TEST']['R2']:.4f} | RMSE: ${metrics['TEST']['RMSE']:,.2f} | MAE: ${metrics['TEST']['MAE']:,.2f}")
print(f"\n  ► Test set R² = {metrics['TEST']['R2']:.4f} (PRIMARY RESULT for papers)")

# Check for overfitting
train_val_diff = metrics['TRAIN']['R2'] - metrics['VAL']['R2']
if train_val_diff > 0.1:
    print(f"\n  ⚠ Warning: Possible overfitting detected (Train R² - Val R² = {train_val_diff:.4f})")
    print('     Consider adjusting alpha/l1_ratio or adding features')
else:
    print(f"\n  ✓ Model generalization looks good (Train R² - Val R² = {train_val_diff:.4f})")

# Coefficients as feature importances
feature_importance = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': np.abs(elastic.coef_)
}).sort_values('Importance', ascending=False)

print(f"\n  ► Top 5 Most Important Features:")
for idx, row in feature_importance.head(5).iterrows():
    print(f"     {row['Feature']:20s}: {row['Importance']:.6f}")

# Export metrics summary to CSV
metrics_df = pd.DataFrame({
    'Dataset': ['Train', 'Validation', 'Test'],
    'R²': [metrics['TRAIN']['R2'], metrics['VAL']['R2'], metrics['TEST']['R2']],
    'MSE': [metrics['TRAIN']['MSE'], metrics['VAL']['MSE'], metrics['TEST']['MSE']],
    'RMSE': [metrics['TRAIN']['RMSE'], metrics['VAL']['RMSE'], metrics['TEST']['RMSE']],
    'MAE': [metrics['TRAIN']['MAE'], metrics['VAL']['MAE'], metrics['TEST']['MAE']]
})

output_file = 'ElasticNet_Regression_Metrics.csv'
metrics_df.to_csv(output_file, index=False)
print(f'\n✓ Metrics saved to: {output_file}')

# Export hyperparameters summary to CSV
hyperparams_df = pd.DataFrame({
    'Hyperparameter': ['alpha', 'l1_ratio', 'max_iter', 'random_state', 'cv_folds', 'cv_r2_score'],
    'Value': [best_params['alpha'], best_params['l1_ratio'], best_params['max_iter'], 42, 5, cv_score],
    'Tuned': ['Yes (GridSearchCV)', 'Yes (GridSearchCV)', 'Yes (GridSearchCV)', 'No (fixed)', 'N/A', 'N/A'],
    'Description': [
        'Regularization strength',
        'Balance between L1 and L2 penalty',
        'Maximum number of iterations',
        'Random seed for reproducibility',
        'Number of cross-validation folds',
        'Best R² score from cross-validation'
    ]
})

hyperparams_file = 'ElasticNet_Hyperparameters.csv'
hyperparams_df.to_csv(hyperparams_file, index=False)
print(f'✓ Hyperparameters saved to: {hyperparams_file}')

# Export feature importances to CSV
feature_importance_file = 'ElasticNet_Feature_Importances.csv'
feature_importance.to_csv(feature_importance_file, index=False)
print(f'✓ Feature importances saved to: {feature_importance_file}')

# Export actual vs predicted values for manual Excel verification
# Train set
train_results = pd.DataFrame({
    'Actual': y_train.values,
    'Predicted': preds_train,
    'Residual': y_train.values - preds_train,
    'Residual_Squared': (y_train.values - preds_train)**2,
    'Absolute_Error': np.abs(y_train.values - preds_train)
})
train_results.to_csv('ElasticNet_Train_Predictions.csv', index=False)

# Validation set
val_results = pd.DataFrame({
    'Actual': y_val.values,
    'Predicted': preds_val,
    'Residual': y_val.values - preds_val,
    'Residual_Squared': (y_val.values - preds_val)**2,
    'Absolute_Error': np.abs(y_val.values - preds_val)
})
val_results.to_csv('ElasticNet_Validation_Predictions.csv', index=False)

# Test set
test_results = pd.DataFrame({
    'Actual': y_test.values,
    'Predicted': preds_test,
    'Residual': y_test.values - preds_test,
    'Residual_Squared': (y_test.values - preds_test)**2,
    'Absolute_Error': np.abs(y_test.values - preds_test)
})
test_results.to_csv('ElasticNet_Test_Predictions.csv', index=False)

print('✓ Train predictions saved to: ElasticNet_Train_Predictions.csv')
print('✓ Validation predictions saved to: ElasticNet_Validation_Predictions.csv')
print('✓ Test predictions saved to: ElasticNet_Test_Predictions.csv')
print('
To manually calculate metrics in Excel:')
print('  MSE  = AVERAGE(Residual_Squared column)')
print('  RMSE = SQRT(MSE)')
print('  MAE  = AVERAGE(Absolute_Error column)')
print('  R²   = RSQ(Actual column, Predicted column)')
print('  or R² = 1 - SUM(Residual_Squared) / DEVSQ(Actual column)')

In [ ]:
# 6) Parity Plot: Actual vs Predicted
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Train parity plot
axes[0].scatter(y_train, preds_train, alpha=0.5, edgecolors='k', linewidth=0.5, color='green')
axes[0].plot([y_train.min(), y_train.max()], [y_train.min(), y_train.max()], 'r--', lw=2, label='Perfect fit')
axes[0].set_xlabel('Actual Weekly Sales ($)', fontsize=12)
axes[0].set_ylabel('Predicted Weekly Sales ($)', fontsize=12)
axes[0].set_title(f"Train Set (R²={metrics['TRAIN']['R2']:.4f})", fontsize=13, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Validation parity plot
axes[1].scatter(y_val, preds_val, alpha=0.5, edgecolors='k', linewidth=0.5)
axes[1].plot([y_val.min(), y_val.max()], [y_val.min(), y_val.max()], 'r--', lw=2, label='Perfect fit')
axes[1].set_xlabel('Actual Weekly Sales ($)', fontsize=12)
axes[1].set_ylabel('Predicted Weekly Sales ($)', fontsize=12)
axes[1].set_title(f"Validation Set (R²={metrics['VAL']['R2']:.4f})", fontsize=13, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Test parity plot
axes[2].scatter(y_test, preds_test, alpha=0.5, edgecolors='k', linewidth=0.5, color='orange')
axes[2].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2, label='Perfect fit')
axes[2].set_xlabel('Actual Weekly Sales ($)', fontsize=12)
axes[2].set_ylabel('Predicted Weekly Sales ($)', fontsize=12)
axes[2].set_title(f"Test Set (R²={metrics['TEST']['R2']:.4f})", fontsize=13, fontweight='bold')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# 7) Model Configuration Summary
print('='*80)
print('ELASTICNET REGRESSION MODEL CONFIGURATION SUMMARY')
print('='*80)

# 1) Selected Features
print('
1. SELECTED FEATURES (11 total):')
print('-'*80)
for i, feat in enumerate(feature_cols, 1):
    print(f'   {i:2d}. {feat}')

# 2) Hyperparameters
print('
2. HYPERPARAMETERS (OPTIMIZED VIA CROSS-VALIDATION):')
print('-'*80)
print('   Model: ElasticNet (sklearn.linear_model.ElasticNet)')
print('   Hyperparameter Optimization:')
print('      - Method: GridSearchCV with 5-fold cross-validation')
print('      - Search space:')
print('          alpha ∈ [0.1, 1.0]')
print('          l1_ratio ∈ [0.2, 0.5, 0.8]')
print('          max_iter ∈ [2000]')
print('      - Scoring metric: R² (coefficient of determination)')
print(f"      - Best alpha: {best_params['alpha']}")
print(f"      - Best l1_ratio: {best_params['l1_ratio']}")
print(f"      - Best max_iter: {best_params['max_iter']}")
print(f"      - CV R² score: {cv_score:.4f}")
print('
   Model Parameters:')
print(f'      - alpha: {elastic.alpha} (OPTIMIZED)')
print(f'      - l1_ratio: {elastic.l1_ratio} (OPTIMIZED)')
print(f'      - max_iter: {elastic.max_iter} (OPTIMIZED)')
print('      - random_state: 42')
print('
   Preprocessing:')
print(